In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load utility functions from detect_clusters.py
import detect_clusters
import matplotlib.pyplot as plt

import skimage
from skimage.io import imread
from skimage.measure import regionprops
import numpy as np
import math
from os.path import join
import pandas as pd
import os
from os.path import join
from pathlib import Path

from detect_clusters import track_cells, matchClusters, detectAndTrackClusters, generate_graph
from pathlib import Path
import os
from skimage.io import imsave
from detect_clusters import computeVelocities, computeRelativeVelocities
from detect_clusters import EdgeFilterAngleCountVelocity
from detect_clusters import labels2rgb, rag_attribute_image, show_rag2
from detect_clusters import rag_velocity_image, vel2rgb
from detect_clusters import to_pandas_nodelist
from detect_clusters import loadDPTracks, EdgeFilterNone

In [3]:
DATADIR = Path('../data/')

## New dataset June 18

In [4]:
from detect_clusters import load_multiframe_labels
from detect_clusters import compute_labelimages_df
from detect_clusters import show_label_df

# COMPUTE SPOTS OURSELVES

# Load image stack as volume (nbframes, height, width)
LL = load_multiframe_labels(DATADIR / 'labelimages.tif')

In [ ]:
# ALTERNATIVE: Lazy loading of the tiff image stack
# Needs: pip install dask zarr tifffile
import dask.array as da
import tifffile as tiff

z = tiff.imread(DATADIR / 'labelimages.tif', aszarr=True)
LL = da.from_zarr(LZ)

In [5]:
# Compute dataframe of regions
ldf = compute_labelimages_df(LL)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:02<00:00, 18.18it/s]


In [6]:
#df0 = pd.read_csv(DATADIR / 'original_spotswithedits.csv', header=0, skiprows=[1,2,3], )
df0 = pd.read_csv(DATADIR / 'original_spots.csv', header=0, skiprows=[1,2,3], )


# Clean up
df = df0[['TRACK_ID','FRAME','POSITION_X','POSITION_Y','ID']]
df = df0.rename(columns = {'TRACK_ID':  'trackid',
                           'FRAME':     'frame',
                           'POSITION_X':'cx',
                           'POSITION_Y':'cy',
                           'ID':        'spotid'})
df = df[['frame','cx','cy','spotid','trackid']]
df = df.sort_values(['frame','trackid'])

#display(df0)
#display(df)

In [35]:
# MERGE TRACKMATER AND LABELIMAGE
from detect_clusters import match_df_to_labeldf

df1, df2 = match_df_to_labeldf(ldf, df, thresh=10.0)

# Use label images as reference, and trackmate to add the trackid info
dfm = pd.merge(df1,df2,how='outer', on=['frame','spotid','trackid','label','match_d'], suffixes=('_TM','')) # TrackMate vs LaBel
# Fill in missing cx,cy from TrackMate
dfm.loc[dfm.label.isna(),['cx','cy']] = dfm.loc[dfm.label.isna(),['cx_TM','cy_TM']]

dfm;

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:00<00:00, 125.28it/s]

Matched label: 6333
Matched trackmate: 6333  (should be equal to previous)
Unmatched label: 191
Unmatched trackmate: 11


In [ ]:
# Optional: add a column for frame diff
detect_clusters.compute_frame_diff_column(dfm);

# frame_diff = 
# nan: start of track
# 0: problem, two cells with same trackid in the frame
# 1: normal
# 2: there was a gap of 1 missed frame

In [34]:
%matplotlib widget

from track_gui import FrameViewer

#rawimage = np.random.rand(*LL.shape) # SHould be actual data

viewer = FrameViewer(dfm, LL, rawimage=None)
viewer.display();

In [376]:
viewer.selected=[]
viewer.update_plot()

In [322]:
viewer.id_fontsize = 6
viewer.update_plot()